# REG 평가 프로그램
- Naive RAG vs Graph RAG

## 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/aiffel/3.rag

In [ ]:
!pwd

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)
print("NEO4J_URI loaded:", os.getenv("NEO4J_URI") is not None)
print("NEO4J_USERNAME loaded:", os.getenv("NEO4J_USERNAME") is not None)
print("NEO4J_PASSWORD loaded:", os.getenv("NEO4J_PASSWORD") is not None)
print("NEO4J_DATABASE loaded:", os.getenv("NEO4J_DATABASE") is not None)

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")


In [ ]:
# YOUR_API_KEY = ''

### 설치:

In [ ]:
!pip install --upgrade langchain-core langchain-community langchain-neo4j
!pip install --upgrade langchain langgraph openai langchain-openai langchain-experimental

In [ ]:
# !pip install langchain langchain-community langgraph neo4j openai

In [ ]:
# !pip install langchain-openai

In [ ]:
# !pip install langchain-experimental

In [ ]:
# !pip install langchain-community langchain-classic

### import

In [ ]:
# from langchain_community.graphs import Neo4jGraph  # 또는 Memgraph
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_experimental.graph_transformers import LLMGraphTransformer
# from langchain_core.documents import Document
# from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

# from langgraph.graph import NetworkGraph  # 새 경로!
# from langchain_community.chains import GraphCypherQAChain
# from langchain_openai import ChatOpenAI
# import os

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#### LLM 설정

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain # Changed import path
from langchain_openai import ChatOpenAI

# OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# test
response = llm.invoke("안녕하세요!")
print(response.content)

### 의존성 모듈 설치

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured

## Graph RAG

#### 클라우드 Neo4j(Aura) + Colab
Neo4j 클라우드에 Colab에서 붙는 방식

- Aura는 Neo4j를 클라우드에서 완전 관리형 서비스(DBaaS)로 제공하는 플랫폼
<details>
<summary>details</summary>
1. Aura가 하는 핵심 역할
- Neo4j 그래프 데이터베이스를 “설치·운영” 걱정 없이 클라우드에서 바로 쓰게 해주는 서비스입니다.

- 서버 인프라, 백업, 패치, 업그레이드, 모니터링 같은 운영을 Neo4j 측이 대신 맡고, 사용자는 Cypher로 쿼리만 날리면 됩니다.

2. 두 가지 주요 서비스 (AuraDB / AuraDS)
- AuraDB: 트랜잭션 그래프 DB 서비스로, 애플리케이션에서 데이터 저장·조회·추천·경로 탐색 같은 그래프 쿼리를 수행할 때 사용합니다.

- AuraDS: Graph Data Science(GDS)용으로, PageRank, 커뮤니티 탐지, 경로 최적화 등 그래프 알고리즘·예측 모델을 돌리기 위한 데이터 사이언스 워크로드에 맞춰진 서비스입니다.
</details>

- [Neo4j Aura 사이트](https://console.neo4j.io)에서 무료 인스턴스 생성

- Aura 콘솔에서 제공하는(자동 생성):
  - Bolt URL
  - username
  - password
를 복사.


#### 1. Neo4j를 python에서 활용 (그래프 DB 연결)

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

transformer = LLMGraphTransformer(llm=llm)

#### 2. Demian.pdf 읽기

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

In [ ]:
pages[0]

In [ ]:
print(pages[10].page_content)

### 3. PDF 로드 + 청크 만들기

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150,)
docs = text_splitter.split_documents(pages)
print(len(docs), docs[0].page_content[:200])

#### 4. Neo4j Aura를 **벡터 스토어(Neo4jVector)**로 사용
- Demian 청크들을 Neo4j Aura에 노드로 저장
- 각 청크에 임베딩을 계산해서 벡터 인덱스까지 Aura 안에 생성
- 따로 Chromadb는 불필요

In [ ]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

url = NEO4J_URI
username = NEO4J_USERNAME
password = NEO4J_PASSWORD

embeddings = OpenAIEmbeddings()

vector_store = Neo4jVector.from_documents(
    documents=docs,
    embedding=embeddings,
    url=url,
    username=username,
    password=password,
    index_name="demian_index",      # 원하는 이름
)


#### 5. RAG 체인 만들기 (Aura만 사용)

In [ ]:
!pip show langchain

In [ ]:
!pip install langchain-classic

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    """당신은 친절한 AI 어시스턴트입니다.
다음 컨텍스트만 사용해서 질문에 답하세요.
모르면 모른다고 말하세요.

컨텍스트:
{context}

질문: {input}"""
)

# 1) 문서 + LLM을 결합하는 체인
question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_variable_name="context",
)

# 2) Retriever와 합쳐서 최종 RAG 체인 구성
qa_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=question_answer_chain,
)

### 6. Demian 내용에 질의 및 답변 저장

In [ ]:
import json

# Demian 질문 5개 + GT (PDF 기반)
questions = [
    "데미안의 작가는 누구인가?",
    "데미안에서 주인공 이름은 무엇인가?",
    "데미안에서 주인공을 괴롭히는 소년 이름은?",
    "데미안에서 케인에 대한 데미안의 해석은 무엇인가?",
    "데미안에서 주인공이 사랑하는 여성 이름은?"
]
ground_truths = [
    "헤르만 헤세 (Hermann Hesse)",
    "에밀 싱클레어 (Emil Sinclair)",
    "프란츠 크로머 (Franz Kromer)",
    "케인은 강하고 용감한 사람으로, 표식은 보호와 구별의 표시이며 약자들은 두려워 왜곡했다",
    "베아트리체 (Beatrice, 또는 Frau Eva)"
]

# 1. qa_chain으로 데이터 자동 수집
data = {
    "question": questions,
    "contexts": [],
    "answer": [],
    "ground_truth": ground_truths
}

print("RAG 실행 중...")
for q in questions:
    result = qa_chain.invoke({"input": q})

    # contexts 추출: list[str]
    contexts = [doc.page_content for doc in result["context"]]
    answer = result["answer"]

    data["contexts"].append(contexts)
    data["answer"].append(answer)

    print(f"Q: {q}\nA: {answer[:100]}...\nContexts: {len(contexts)}개\n")

# 저장
with open("demian_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("저장 완료: demian_rag_results.json")

- 간단한 테스트

In [ ]:
# query = "데미안에서 싱클레어가 처음으로 '표식(mark)'에 대해 깨닫는 장면이 뭐야?"
# result = qa_chain.invoke({"input": query})

# print("답변:\n", result["answer"])
# print("\n[출처 페이지들]")
# for doc in result["context"]:
#     print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

In [ ]:
# 사용형식
# result = qa_chain.invoke({"input": "여기에 질문"})
# print(result["answer"])          # 최종 답변
# print(result["context"])         # 사용된 문서들

## 기존 Naive REG

#### 1. 토큰 준비

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

#### 2. 문서 읽기

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 위(Graph REG)에서 기 사용
# loader = PyPDFLoader("Demian.pdf")
# pages = loader.load_and_split()

n_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, length_function = tiktoken_len)
n_docs = n_text_splitter.split_documents(pages)

#### 3. TextEmbedding

In [ ]:
# OpenAI embedding 사용
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 생성 (추천: text-embedding-3-small / large)
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 또는 "text-embedding-3-large"
)

In [ ]:
# test
text = "임베딩 테스트 문장입니다."
vector = embedding_model.embed_query(text)
print(len(vector), vector[:5])  # 벡터 길이, 일부만 출력

#### 4. VectorStore 사용 : ChromaDB

In [ ]:
!pip check

In [ ]:
%%writefile constraints.txt
opentelemetry-sdk==1.38.0
opentelemetry-api==1.38.0
opentelemetry-proto==1.38.0
opentelemetry-exporter-otlp-proto-common==1.38.0
langgraph==1.0.2
requests==2.32.5
fsspec==2024.12.0
jedi>=0.18

!pip install -c constraints.txt chromadb langchain-neo4j neo4j-graphrag unstructured

!pip check

In [ ]:
# !pip install -c constraints.txt chromadb

In [ ]:
#!pip install langchain-chroma

In [ ]:
from langchain_community.vectorstores import Chroma

- Chroma에 임베딩

In [ ]:
db = Chroma.from_documents(n_docs, embedding_model)


#### 5. Retriever 사용  

- Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할

- 사용자 쿼리 → Retriever(ChromaDB 검색) → 관련 청크 반환 → LLM에 컨텍스트+쿼리 전달 → 답변 생성

In [ ]:
!pip install -q langchain-classic


In [ ]:
from langchain_classic.chains import RetrievalQA


In [ ]:
# from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# llm = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

In [ ]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI

llm2 = ChatOpenAI(
    model="gpt-4o-mini",  # 또는 "gpt-4o"
    temperature=0.0,
    streaming=True,  # 스트리밍 활성화
    callbacks=[StreamingStdOutCallbackHandler()]  # 실시간 출력
)

In [ ]:
from langchain_core.prompts import PromptTemplate

# 커스텀 프롬프트 (한국어)
prompt_template = """당신은 친절한 AI 어시스턴트입니다.
다음 컨텍스트만 사용해서 질문에 답하세요.
모르면 모른다고 말하세요.

컨텍스트:
{context}

질문: {question}
답변:"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

# qa = RetrievalQA.from_chain_type(llm2, chain_type="stuff",
#                                  retriever=db.as_retriever(
#                                      search_type="mmr",
#                                      search_kwargs={"k": 3, "fetch_k" : 10}),
#                                  return_source_documents=True)

qa = RetrievalQA.from_chain_type(
    llm=llm2,
    chain_type="stuff",
    retriever=db.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    ),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}  # 커스텀 프롬프트 적용!
)

<details>
<summary> 파라미터 설명 </summary>

🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.
</detatils>

#### 6. Demian 내용에 질의 및 답변 저장

In [ ]:
import json

# Demian 질문 5개
questions = [
    "데미안의 작가는 누구인가?",
    "데미안에서 주인공 이름은 무엇인가?",
    "데미안에서 주인공을 괴롭히는 소년 이름은?",
    "데미안에서 케인에 대한 데미안의 해석은 무엇인가?",
    "데미안에서 주인공이 사랑하는 여성 이름은?"
]
ground_truths = [
    "헤르만 헤세 (Hermann Hesse)",
    "에밀 싱클레어 (Emil Sinclair)",
    "프란츠 크로머 (Franz Kromer)",
    "케인은 강하고 용감한 사람으로, 표식은 보호와 구별의 표시이며 약자들은 두려워 왜곡했다",
    "베아트리체 (Beatrice, 또는 Frau Eva)"
]

# 데이터 수집 (위 Graph REG 구조와 호환)
data = {
    "question": questions,
    "contexts": [],
    "answer": [],
    "ground_truth": ground_truths
}

print("RAG 실행 중...")
for i, q in enumerate(questions):
    result = qa(q)  # 간단한 호출

    # 기존 result 구조: result["result"], result["source_documents"]
    answer = result["result"]
    contexts = [doc.page_content for doc in result["source_documents"]]

    data["contexts"].append(contexts)
    data["answer"].append(answer)

    print(f"Q{i+1}: {q}")
    print(f"A: {answer[:100]}...")
    print(f"Contexts: {len(contexts)}개\n")

print("=== 평가 데이터 수집 완료 ===")
print("data 딕셔너리:", list(data.keys()))

# 저장
with open("demian_naive_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("저장 완료: demian_naive_rag_results.json")

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [ ]:
# from langchain_openai import ChatOpenAI

# # llm3 = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)
# llm3 = ChatOpenAI(
#     model="gpt-4o-mini",  # 또는 "gpt-4o"
#     temperature=0.0
# )

# request = llm2.invoke("how demian looks like")
# display(Markdown(request.content))


# RAGAS 평가

### 평가준비

#### 답변 파일 읽기

In [ ]:
# 답변 로드해서 Dataset 만들기
from datasets import Dataset
import json

with open("demian_rag_results.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

GraphREG_dataset = Dataset.from_dict(loaded_data)
print(GraphREG_dataset)

print("-" * 50)

with open("demian_naive_rag_results.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

NaiveREG_dataset = Dataset.from_dict(loaded_data)
print(NaiveREG_dataset)


In [ ]:
# test
from pprint import pprint

one_row_dataset = GraphREG_dataset.select([0])
pprint(one_row_dataset[0])

print("-" * 50)

one_row_dataset = NaiveREG_dataset.select([0])
pprint(one_row_dataset[0])


#### 환경 준비

In [ ]:
# [실습 환경 준비]
# Ragas 및 LangChain 관련 최신 패키지를 설치합니다.
# !pip install -qU ragas langchain langchain-openai langchain-community chromadb tiktoken nest_asyncio pandas

!pip install -qU \
  ragas==0.1.14 \
  langchain==0.2.14 \
  langchain-openai==0.1.23 \
  langchain-community==0.2.12 \
  chromadb==1.5.5 \
  tiktoken==0.7.0 \
  nest_asyncio==1.6.0 \
  pandas==2.2.2 \
  requests==2.32.4 \
  fsspec==2024.12.0

!pip check



In [ ]:
# RAGAS는 평가 속도를 높이기 위해 내부적으로 asyncio.gather를 사용해 수백 개의 API를 동시에 비동기 호출합니다.
# Jupyter/Colab 환경은 이미 자체적인 비동기 이벤트 루프를 돌리고 있어서 충돌이 발생합니다.
# nest_asyncio는 이 충돌을 막아주어 Colab에서도 RAGAS가 병렬 처리를 할 수 있게 해줍니다.
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import os
from google.colab import userdata

# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [ ]:
# 판사(Judge) 역할을 할 LLM과 임베딩 세팅
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### Metics 정의 및 평가

In [ ]:
import ragas
from ragas import metrics

print(metrics.__all__)
# 또는 dir(metrics)로 어떤 이름이 있는지 확인
print(dir(metrics))

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextRecall,
    ContextPrecision,
)

metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    ContextRecall(),
    ContextPrecision(),
]

# 결과 평가
graph_result = evaluate(
    dataset=GraphREG_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False # API 에러 발생 시 중단하지 않음
)

naive_result  = evaluate(
    dataset=NaiveREG_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False # API 에러 발생 시 중단하지 않음
)


In [ ]:
print(naive_result)

### 평가 결과

#### 평균 점수 비교

In [ ]:
# pd.set_option('display.max_colwidth', None)
# df_result = result.to_pandas()
# display(df_result[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']])

# 메트릭별 비교 (class 이름 기준)
metric_keys = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("\n=== 메트릭별 평균 점수 비교 ===")
for key in metric_keys:
    g_score = graph_result[key]
    n_score = naive_result[key]
    print(f"{key:18s} | GraphREG: {g_score:.3f} | NaiveREG: {n_score:.3f}")

<details>
<summary>각 지표가 의미하는 것</summary>

- faithfulness (충실도)
  - 생성 답변이 실제로 제공된 컨텍스트에 얼마나 거짓 없이 근거를 두고 있는지 보는 지표.

  - 0.2라는 값은 둘 다 “컨텍스트에 없는 말을 하거나 애매한 부분이 꽤 있다”는 뜻으로, 두 모델이 충실도 측면에서는 거의 동급이라는 의미다.

- answer_relevancy (답변 관련성)
  - 질문 의도에 답변이 얼마나 직접적으로 관련 있는지 평가.
```
GraphREG: 0.105
NaiveREG: 0.031
```
  - 절대값 자체는 낮지만, GraphREG가 NaiveREG보다 질문과 더 맞는 방향으로 대답하는 경향이 있다.
   예: 질문이 “표식에 대해 깨닫는 장면”인데, GraphREG는 그 장면 근처를 설명하고, Naive는 소설 전체 분위기만 말하는 식.

- context_precision (컨텍스트 정밀도)
  - 검색된 컨텍스트들 중 “실제 답변에 유용한 것들 비율”을 보는 지표.
```
GraphREG: 0.000
NaiveREG: 0.267
```
  - 여기서는 오히려 NaiveREG가 더 적절한 문단을 골라온 것처럼 평가된 상황이다.
  - GraphREG의 precision=0.0은, RAGAS 기준으로 “답변에 쓰인 핵심 정보가 검색 컨텍스트에 잘 안 보인다” 혹은 “잡소스가 많고, 핵심 문단이 위에 안 떠 있다”라는 해석이 가능하다.

- context_recall (컨텍스트 재현율)
질문에 답하는 데 필요한 정보가 검색된 컨텍스트 안에 충분히 들어 있는지 보는 지표.
```
GraphREG: 0.400
NaiveREG: 0.200
```
  - GraphREG는 답변에 필요한 정보는 더 많이 포함하지만, 그중에서 정말 필요한 부분만 깔끔하게 골라 쓰지는 못해 precision이 0이 되는 그림이라고 볼 수 있다.
  - 반대로 NaiveREG는 적지만 꽤 관련 있는 chunk만 가져와서 recall은 낮고 precision은 높은 패턴.

- 요약해서 해석
  - 둘 다 faithfulness=0.2로 “데미안 PDF 컨텍스트에 충실한 답변을 꾸준히 내는 수준까지는 아직 부족”하다.

  - GraphREG

    - 질문 관련성(답변 내용)이 더 좋음: answer_relevancy ↑

    - 필요한 정보는 더 많이 가져오지만(context_recall ↑), 거기서 실제 답변에 쓴 “핵심 근거 chunk”가 RAGAS 기준으로 잘 매핑되지 않아 precision=0.

  - NaiveREG

    - 질문 관련성은 떨어지지만(answer_relevancy 낮음),

    - 적은 양의 컨텍스트 중에서 “질문과 직접 관련 있는 문단”을 상대적으로 잘 뽑은 것으로 평가(context_precision > 0).

  - 그래서 현재 세팅에서는:

    -“사용자 질문에 더 잘 맞는 답을 주냐?” → GraphREG가 우세.

    - “적은 컨텍스트로도 딱 맞는 문단을 뽑았냐?” → NaiveREG가 더 나아 보임.

  - 실무적으로는

  1. GraphREG의 리트리버를 좀 더 aggressive하게 정리해서 불필요한 chunk 줄이기(precision ↑),

  2. NaiveREG는 질문 의도에 맞는 답변을 더 직접적으로 생성하도록 프롬프트/LLM 튜닝(AnswerRelevancy ↑)
방향으로 튜닝하면 두 시스템 모두 개선 여지가 있다.

</details>

#### 평가 상세

In [ ]:
from pprint import pprint

# 1) 결과를 dict/list 형태로 변환 (버전에 따라 약간 다를 수 있음)
graph_df = graph_result.to_pandas()
naive_df = naive_result.to_pandas()

# graph_df / naive_df에는 각 row가 하나의 질문에 해당하고,
# 컬럼 이름은 'question', 'answer', 'faithfulness', 'answer_relevancy', ... 형태라고 가정.
print(graph_df.columns)
print(naive_df.columns)

# 2) 질문 수 확인 (둘이 동일해야 함)
num_rows = len(GraphREG_dataset)
assert num_rows == len(NaiveREG_dataset)

# 3) 각 질문에 대한 상세 정보 출력
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

for i in range(num_rows):
    q = GraphREG_dataset[i]["question"]          # 두 dataset 모두 question 동일하다고 가정
    g_ans = GraphREG_dataset[i]["answer"]
    n_ans = NaiveREG_dataset[i]["answer"]

    print("=" * 80)
    print(f"[Q{i+1}] 질문: {q}\n")

    print("[GraphREG 답변]")
    print(g_ans, "\n")

    print("[NaiveREG 답변]")
    print(n_ans, "\n")

    # 점수는 graph_df / naive_df에서 같은 인덱스 row에서 꺼냄
    print("[평가 점수]")
    for col in metric_cols:
        if col in graph_df.columns and col in naive_df.columns:
            g_score = graph_df.iloc[i][col]
            n_score = naive_df.iloc[i][col]
            print(f"  - {col}: GraphREG={g_score:.3f}, NaiveREG={n_score:.3f}")
    print()  # blank line
